# Project A — Load DataCo Smart Supply Chain Data & Run Integrity Checks
**Steps 1-3 of the pipeline:** load the raw CSV, profile it, stage it into a local SQLite DB, and run the integrity checks.

Dataset: [DataCo Smart Supply Chain for Big Data Analysis](https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis)

**Before running:** download the CSV from Kaggle and update `CSV_PATH` below. Also run `pd.read_csv(CSV_PATH, encoding='latin-1').columns.tolist()` once to confirm headers match `COLUMN_MAP` — Kaggle re-uploads of this dataset have had minor header-naming variations.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

CSV_PATH = "C:/Users/Dell/Desktop/EYS/Project1/data/DataCoSupplyChainDataset.csv"   # <-- update this
DB_PATH = "dataco_staging.db"
CHECKS_SQL_PATH = "C:/Users/Dell/Desktop/EYS/Project1/sql/02_integrity_checks.sql"  # optional, only needed for the last section

## 1. Column mapping
Raw DataCo headers -> our staging schema names. We also drop PII columns (email/password/name/street/zip) on load — they add no modeling value and shouldn't be staged.

In [2]:
COLUMN_MAP = {
    "Type": "order_type",
    "Days for shipping (real)": "days_for_shipping_real",
    "Days for shipment (scheduled)": "days_for_shipment_scheduled",
    "Benefit per order": "benefit_per_order",
    "Sales per customer": "sales_per_customer",
    "Delivery Status": "delivery_status",
    "Late_delivery_risk": "late_delivery_risk",
    "Category Id": "category_id",
    "Category Name": "category_name",
    "Customer City": "customer_city",
    "Customer Country": "customer_country",
    "Customer Id": "customer_id",
    "Customer Segment": "customer_segment",
    "Customer State": "customer_state",
    "Department Id": "department_id",
    "Department Name": "department_name",
    "Latitude": "latitude",
    "Longitude": "longitude",
    "Market": "market",
    "Order City": "order_city",
    "Order Country": "order_country",
    "Order Customer Id": "order_customer_id",
    "order date (DateOrders)": "order_date",
    "Order Id": "order_id",
    "Order Item Cardprod Id": "order_item_cardprod_id",
    "Order Item Discount": "order_item_discount",
    "Order Item Discount Rate": "order_item_discount_rate",
    "Order Item Id": "order_item_id",
    "Order Item Product Price": "order_item_product_price",
    "Order Item Profit Ratio": "order_item_profit_ratio",
    "Order Item Quantity": "order_item_quantity",
    "Sales": "sales",
    "Order Item Total": "order_item_total",
    "Order Profit Per Order": "order_profit_per_order",
    "Order Region": "order_region",
    "Order State": "order_state",
    "Order Status": "order_status",
    "Product Card Id": "product_card_id",
    "Product Category Id": "product_category_id",
    "Product Name": "product_name",
    "Product Price": "product_price",
    "Product Status": "product_status",
    "shipping date (DateOrders)": "shipping_date",
    "Shipping Mode": "shipping_mode",
}

DROP_COLUMNS_CONTAINING = ["email", "password", "fname", "lname", "street", "zipcode"]

## 2. Load and clean the CSV

In [4]:
def load_csv(csv_path: str) -> pd.DataFrame:
    # DataCo's CSV is commonly latin-1 encoded, not utf-8
    df = pd.read_csv(csv_path, encoding="latin-1", low_memory=False)

    pii_cols = [c for c in df.columns if any(flag in c.lower() for flag in DROP_COLUMNS_CONTAINING)]
    df = df.drop(columns=pii_cols, errors="ignore")
    print(f"Dropped {len(pii_cols)} PII columns: {pii_cols}")

    available_map = {k: v for k, v in COLUMN_MAP.items() if k in df.columns}
    missing = set(COLUMN_MAP) - set(available_map)
    if missing:
        print(f"WARNING: {len(missing)} expected columns not found: {missing}")
        print("  -> check actual headers with df.columns.tolist()")

    df = df[list(available_map.keys())].rename(columns=available_map)

    for date_col in ("order_date", "shipping_date"):
        if date_col in df.columns:
            df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    return df

df = load_csv(CSV_PATH)
df.head()

Dropped 7 PII columns: ['Customer Email', 'Customer Fname', 'Customer Lname', 'Customer Password', 'Customer Street', 'Customer Zipcode', 'Order Zipcode']


,order_type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,...,order_region,order_state,order_status,product_card_id,product_category_id,product_name,product_price,product_status,shipping_date,shipping_mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,Southeast Asia,Java Occidental,COMPLETE,1360,73,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,South Asia,Rajastán,PENDING,1360,73,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,South Asia,Rajastán,CLOSED,1360,73,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,Oceania,Queensland,COMPLETE,1360,73,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,Oceania,Queensland,PENDING_PAYMENT,1360,73,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class


## 3. Profile the data
Row counts, null counts, duplicates, implausible values — this output is your evidence for the README and for defending the project in interview.

In [5]:
print("--- Basic profile ---")
print(f"Rows: {len(df):,}")
print(f"Distinct orders: {df['order_id'].nunique():,}")
print(f"Distinct products: {df['product_card_id'].nunique():,}")
print(f"Distinct regions: {df['order_region'].nunique()}")
print(f"Date range: {df['order_date'].min()} to {df['order_date'].max()}")

--- Basic profile ---
Rows: 180,519
Distinct orders: 65,752
Distinct products: 118
Distinct regions: 23
Date range: 2015-01-01 00:00:00 to 2018-01-31 23:38:00


In [6]:
key_cols = [
    "order_date", "shipping_date", "order_item_quantity",
    "product_price", "days_for_shipment_scheduled",
    "days_for_shipping_real", "order_region",
]
print("--- Null counts (key columns) ---")
df[key_cols].isna().sum()

--- Null counts (key columns) ---


order_date                     0
shipping_date                  0
order_item_quantity            0
product_price                  0
days_for_shipment_scheduled    0
days_for_shipping_real         0
order_region                   0
dtype: int64

In [7]:
dupe_count = df["order_item_id"].duplicated().sum()
bad_qty = (df["order_item_quantity"] <= 0).sum()
bad_price = (df["product_price"] < 0).sum()
ship_before_order = (df["shipping_date"] < df["order_date"]).sum()

print(f"Duplicate order_item_id rows: {dupe_count:,}")
print(f"Non-positive quantity rows: {bad_qty:,}")
print(f"Negative price rows: {bad_price:,}")
print(f"Shipped-before-ordered rows: {ship_before_order:,}")

Duplicate order_item_id rows: 0
Non-positive quantity rows: 0
Negative price rows: 0
Shipped-before-ordered rows: 0


## 4. Load into the staging database (SQLite)
Swap the engine string in `get_engine()` for MySQL/SQL Server later — every query written against this stays standard SQL.

In [8]:
def get_engine(db_path: str = DB_PATH):
    return create_engine(f"sqlite:///{db_path}")

engine = get_engine()
with engine.begin() as conn:
    df.to_sql("stg_dataco_raw", conn, if_exists="replace", index=False)

print(f"Loaded {len(df):,} rows into stg_dataco_raw ({DB_PATH})")

Loaded 180,519 rows into stg_dataco_raw (dataco_staging.db)


## 5. Run the integrity-check SQL against the staged table
Executes each statement in `sql/02_integrity_checks.sql` and prints the results inline — no need to leave the notebook to run the checks.

In [10]:
import re

def run_integrity_checks(engine, sql_file: str):
    sql_text = Path(sql_file).read_text()

    # Strip inline/full-line comments BEFORE splitting on ';' — comments in
    # this file contain semicolons inside plain-English sentences (e.g.
    # "should be unique; flag any that aren't"), which breaks a naive
    # split-on-';' parser. Removing everything from '--' to end-of-line
    # first avoids that trap entirely.
    lines_no_comments = [re.sub(r"--.*$", "", line) for line in sql_text.splitlines()]
    sql_no_comments = "\n".join(lines_no_comments)

    statements = [s.strip() for s in sql_no_comments.split(";") if s.strip()]

    with engine.connect() as conn:
        for i, stmt in enumerate(statements, start=1):
            try:
                result = pd.read_sql(text(stmt), conn)
                print(f"=== Check {i} ===")
                display(result.head(10))
                if len(result) > 10:
                    print(f"... ({len(result)} rows total)")
            except Exception as e:
                print(f"=== Check {i} failed: {e} ===")

## Next steps
- Build `dim_sku.unit_cost` from `product_price`, classify SKU velocity (fast/medium/slow)
- Fit demand and lead-time-deviation distributions per SKU-velocity class (`scipy.stats`)
- Build the (Q,R) reorder policy + PuLP optimizer + Monte Carlo simulation